# 29장 · 체크포인터로 대화를 이어가고 분리하기

대출 기간을 질문한 사람이 곧바로 반납 방법을 묻습니다. 다음 요청에서 앞선 대화를 쓰려면 메시지 목록을 병합하는 규칙과 대화별 저장 위치가 모두 필요합니다.

실행을 확인한 커널은 Python 3.12.15입니다. 예제에 사용한 외부 패키지는 langchain-core 1.6.12, langgraph 1.2.16, langgraph-checkpoint 4.3.0입니다.

Python 3 커널을 선택하고 위에서부터 실행하세요. 과제 셀에 직접 코드를 작성한 뒤 바로 아래에서 풀이를 펼쳐 보세요.

## 첫 질문이 저장된 곳 찾기

체크포인터는 그래프가 실행되는 동안 상태를 보관합니다. thread_id가 어느 대화의 상태를 읽고 쓸지 가리킵니다. 노트북에서는 그래프 하나와 메모리 저장소 하나를 만들어 여러 번 사용합니다.



**볼 곳**　‘첫 질문’를 ‘대화별 저장’에 남기는 시점과 ‘이어서 읽기’를 확인하는 시점을 구분하세요.

**애니메이션의 의미**　같은 대화 ID를 써야 이전 질문을 이어서 읽습니다.


### 셀 01 · A 대화 열기

제공 함수는 가상 답변을 만들어 실제 LangGraph의 InMemorySaver에 저장합니다. 그래프는 이 셀에서 한 번만 만드세요.

In [1]:
from typing import Annotated, TypedDict
from langgraph.graph import StateGraph, START, END
from langchain_core.messages import HumanMessage, AIMessage, AnyMessage
from langgraph.graph.message import add_messages
from uuid import uuid4
from langgraph.checkpoint.memory import InMemorySaver

class State(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]

def respond(state: State) -> dict:
    humans = [m for m in state["messages"] if isinstance(m, HumanMessage)]
    if not humans:
        raise ValueError("사용자 메시지가 필요합니다.")
    content = f"이 대화의 사용자 발화 {len(humans)}개. 최근 질문: {humans[-1].content}"
    return {"messages": [AIMessage(content=content, id=str(uuid4()))]}

def make_graph():
    builder = StateGraph(State)
    builder.add_node("respond", respond)
    builder.add_edge(START, "respond")
    builder.add_edge("respond", END)
    return builder.compile(checkpointer=InMemorySaver())

def config_for(thread_id: str) -> dict:
    if not thread_id.strip():
        raise ValueError("대화 ID가 필요합니다.")
    return {"configurable": {"thread_id": thread_id}}

def send(graph, thread_id: str, question: str):
    if not question.strip():
        raise ValueError("질문이 비어 있습니다.")
    update = {"messages": [HumanMessage(content=question, id=str(uuid4()))]}
    return graph.invoke(update, config=config_for(thread_id))

graph = make_graph()
reply = send(graph, 'A', '대출 기간?')
print(reply['messages'][-1].content)

이 대화의 사용자 발화 1개. 최근 질문: 대출 기간?


사용자 발화가 1개로 나옵니다. respond가 이 대화 안의 HumanMessage만 세므로 AI 답변은 개수에 들어가지 않습니다.

### 셀 02 · 같은 대화 이어 쓰기

이미 저장된 첫 질문을 다시 넘기지 말고 두 번째 질문만 보냅니다.

In [2]:
reply = send(graph, 'A', '반납 방법?')
print(reply['messages'][-1].content)

이 대화의 사용자 발화 2개. 최근 질문: 반납 방법?


사용자 발화 2개가 되었습니다. 같은 저장소와 같은 A를 썼기 때문입니다. 중간에 첫 셀을 다시 실행하면 저장소를 새로 만들어 과거 상태가 사라집니다.

## 새 대화가 읽는 상태

학생 두 명에게 같은 대화 ID를 주면 기록이 섞일 수 있습니다. 새 대화 ID와 기존 대화 ID는 서로 다른 상태를 가리킵니다. InMemorySaver의 기록은 현재 프로세스가 끝나면 사라집니다.



**볼 곳**　기존 대화의 질문과 새 대화의 첫 질문을 나란히 읽으세요.

**애니메이션의 의미**　새 대화는 다른 대화의 질문을 읽지 않습니다.


### 같은 저장소에서 대화 A와 B 따라가기



**볼 곳**　같은 저장소에서 대화 A와 B가 각각 어떤 질문을 다시 읽는지 보세요.

**애니메이션의 의미**　체크포인트는 대화 식별자별로 조회합니다. 같은 저장소를 써도 대화 내용은 분리됩니다.


### 과제 03 · B 대화의 첫 질문 세기

B 대화에 ‘휴관일?’을 보내 마지막 답의 발화 수를 확인하세요.

A의 두 번의 왕복은 사용자 → 답변 → 사용자 → 답변 순입니다. 전체 메시지는 4개, 그중 사용자 질문은 2개입니다. B에서 첫 질문 수가 1이 되어도 A의 네 메시지는 같은 저장소의 다른 ID 아래 남아 있습니다.

In [3]:
# graph는 그대로 사용하고 대화 ID만 B로 바꾸세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>reply_b = send(graph, &#x27;B&#x27;, &#x27;휴관일?&#x27;)
print(reply_b[&#x27;messages&#x27;][-1].content)</code></pre>
<p>실행 결과</p><pre><code>이 대화의 사용자 발화 1개. 최근 질문: 휴관일?</code></pre>
<p>A에 두 번 질문했어도 B의 첫 질문은 1개입니다. B에서 3개라면 호출 때 전달한 ID부터 살펴보세요.</p>
</details>

## 돌아보기

graph.get_state(config_for('A')).values에서 메시지 수를 예측해 보세요. 두 발화에 각각 답이 붙어 총 네 개입니다. 이미 실행한 셀을 반복하면 그만큼 새 발화가 더해집니다.